# Kernel density estimation (KDE)

Fits a Gaussian kernel density surface (bandwidth 500 m) to each point dataset and evaluates it at the population grid centroids, producing a normalised relative density (`kde_rel`) per cell for comparison across masking methods.

**Input:** the point datasets in `POINT_FILES` and the population grid.
**Output:** one shapefile per dataset in `Kernel-Density-Estimation/`.


In [ ]:
# Imports
import os
from pathlib import Path
import numpy as np
import geopandas as gpd
from sklearn.neighbors import KernelDensity


In [ ]:
#Paths
# Data directory. Override with the DATA_DIR environment variable to
# point at a different copy of the data without editing this file, e.g.
#   DATA_DIR=/path/to/data jupyter execute this_notebook.ipynb
# The data itself is not part of this repository -- see README.md.
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))

POINT_FILES = {
     "original_points": DATA_DIR / "original" / "2008_eov_illness_stand.shp",
    "donut_150_900": DATA_DIR / "donut_masking" / "donut_points150_high900.shp",
    "stat_masked": DATA_DIR / "statistical_masking" / "statmasked_points.shp",
    "synthetic_sdv": DATA_DIR / "synthetic" / "synthetic_points.shp",
    "areal_grid_release": DATA_DIR / "area_aggregation" / "Quasi_points.shp",
}


GRID_PATH = DATA_DIR / "population_grid" / "GHS_Hungary_pop_city-clipped2.shp"


OUT_DIR = Path("Kernel-Density-Estimation")
OUT_DIR.mkdir(parents=True, exist_ok=True)


#Parameters
CRS_EPSG = 23700
BANDWIDTH = 500

In [ ]:
#Loading grid and computing centroids for KDE evaluation
grid = gpd.read_file(GRID_PATH).to_crs(epsg=CRS_EPSG)
eval_coords = np.column_stack([grid.geometry.centroid.x, grid.geometry.centroid.y])

#Performing KDE for each point file and saving results
for name, path in POINT_FILES.items():
    print(f"Processing {name}")

    #Loading points and extracting coordinates
    points = gpd.read_file(path).to_crs(epsg=CRS_EPSG)
    coords = np.column_stack([points.geometry.x, points.geometry.y])

    kde = KernelDensity(kernel="gaussian", bandwidth=BANDWIDTH)
    kde.fit(coords)

    #Evaluating KDE on grid centroids and computing relative values
    kde_values = np.exp(kde.score_samples(eval_coords))

    out = grid.copy()
    out["kde"] = kde_values
    out["kde_rel"] = kde_values / kde_values.max()

    out.to_file(OUT_DIR / f"{name}_kde.shp", driver="Esri Shapefile")